In [ ]:
pip install ftfy

In [ ]:
import pandas as pd
import requests
import numpy as np
import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt
import json
from bs4 import BeautifulSoup
import requests
import math
import scipy.stats as stats
import random
import datetime
import unicodedata
import re
import string
from geopy.geocoders import Nominatim

In [ ]:
# Affichage complet colonnes, observations, etc.
pd.set_option('display.max_columns', None)  # Affiche toutes les colonnes
pd.set_option('display.max_rows', None)     # (optionnel) Affiche toutes les lignes si nécessaire
pd.set_option('display.width', None)        # Pas de coupure de ligne
pd.set_option('display.max_colwidth', None) # Affiche tout le contenu des cellules

In [ ]:
def corriger_et_nettoyer_texte(texte):
    if not isinstance(texte, str):
        return texte  # Ne traite que les chaînes

    # 1. Correction d'encodage
    try:
        texte = texte.encode('raw_unicode_escape').decode('utf-8')
    except Exception:
        try:
            texte = texte.encode('latin1', errors='ignore').decode('utf-8', errors='ignore')
        except Exception:
            pass  # On garde le texte tel quel

    # 2. Nettoyage des artefacts type Word/Excel
    texte = texte.replace('_x000D_', ' ')

    # 2.1. Suppression des séquences encodées de type _xHHHH_
    texte = re.sub(r'_x[0-9A-Fa-f]{4}_', '', texte)

    # 3. Suppression de caractères invisibles ou bizarres
    texte = re.sub(r'[\u00A0\u200B\u2028\u2029\r\n]+', ' ', texte)

    # 4. Nettoyage des espaces doubles
    texte = re.sub(r'\s+', ' ', texte)

    return texte.strip()

## Et tu l’appliques comme ceci :

## Modifier
## df["domaine"] = df["domaine"].apply(corriger_et_nettoyer_texte)

In [ ]:
# Fonction traitement des faux NaN
#je crée une liste de faux nul, ce qui permettra par la suite de pouvoir les traiter
faux_nans = ['NaN', 'nan', 'None', 'NULL', 'null', 'na', '-', '--', '', ' ', '\\n']
def detecter_faux_nan(df, valeurs_fausses=faux_nans):   #on crée la fonction afin d'identifier les nans
    faux_nan_counts = {}

    for col in df.columns:
        # On travaille sur des chaînes nettoyées (strip) et converties en str
        faux = df[col].astype(str).str.strip().isin(valeurs_fausses)
        nb_faux = faux.sum()
        if nb_faux > 0:
            faux_nan_counts[col] = nb_faux

    return pd.Series(faux_nan_counts).sort_values(ascending=False)

In [ ]:
## Lecture de la table domaine
df_app_region = pd.read_csv('/content/Lien_Appelation_Region.csv')
df_sous_region = pd.read_csv('/content/Liste_Sous_Region.csv')
df_appellation = pd.read_csv('/content/Liste_Appellation.csv')
df_region = pd.read_csv('/content/Liste_Region.csv')

In [ ]:
df_region